In [2]:
import pandas as pd
import re


In [3]:
def normalize_name(name):
    if not isinstance(name, str):
        return None

    name = name.lower()
    name = re.sub(r"[^a-z\s]", "", name)
    name = re.sub(r"\s+", " ", name).strip()

    parts = name.split()

    # collapse spaced initials: "s p narine" -> "sp narine"
    if len(parts) >= 3 and all(len(p) == 1 for p in parts[:-1]):
        name = "".join(parts[:-1]) + " " + parts[-1]

    return name


In [4]:
def initial_match(n1, n2):
    t1 = n1.split()
    t2 = n2.split()

    if len(t1) < 2 or len(t2) < 2:
        return False

    # last name must match
    if t1[-1] != t2[-1]:
        return False

    f1, f2 = t1[0], t2[0]

    # handle single or multi initials: v, ms, sp, ab
    if len(f1) <= 3:
        return f2.startswith(f1[0])
    if len(f2) <= 3:
        return f1.startswith(f2[0])

    return False


In [5]:
# Base master list
base = pd.read_csv("outputs/player_total_value_master.csv")

# Spaced initials dataset
spaced = pd.read_csv("data_processed/cricketers.csv")

# Full-name dataset
full = pd.read_csv("data_processed/players_data_with_all_info.csv")


In [6]:
spaced["name_norm"] = spaced["Name"].apply(normalize_name)

spaced_ref = (
    spaced
    .dropna(subset=["Country"])
    .groupby("name_norm")["Country"]
    .agg(lambda x: x.mode()[0])
)


In [7]:
full["fullname_norm"] = full["fullname"].apply(normalize_name)

full_ref = (
    full
    .dropna(subset=["country_name"])
    .groupby("fullname_norm")["country_name"]
    .agg(lambda x: x.mode()[0])
)


In [ ]:
def infer_nationality_from_names(player_name):
    pn = normalize_name(player_name)

    #  Exact match — spaced initials
    if pn in spaced_ref:
        return spaced_ref[pn], "spaced_initials"

    #  Initial match — spaced initials
    for ref_name, country in spaced_ref.items():
        if initial_match(pn, ref_name):
            return country, "spaced_initials"

    #  Exact match — full names
    if pn in full_ref:
        return full_ref[pn], "full_name"

    #  Initial match — full names
    for ref_name, country in full_ref.items():
        if initial_match(pn, ref_name):
            return country, "full_name"

    return "unknown", "none"


In [9]:
base[["inferred_nationality", "nationality_source"]] = base["player"].apply(
    lambda x: pd.Series(infer_nationality_from_names(x))
)


In [10]:
base.to_csv(
    "outputs/player_total_value_master_enriched.csv",
    index=False
)


In [11]:
base["nationality_source"].value_counts()


nationality_source
spaced_initials    1471
full_name          1356
none                731
Name: count, dtype: int64

In [12]:
full["fullname_norm"] = full["fullname"].apply(normalize_name)

style_ref = (
    full
    .dropna(subset=["battingstyle", "bowlingstyle"], how="all")
    .groupby("fullname_norm")
    .agg({
        "battingstyle": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        "bowlingstyle": lambda x: x.mode().iloc[0] if not x.mode().empty else None
    })
)


In [13]:
def infer_styles_from_names(player_name):
    pn = normalize_name(player_name)

    # Exact match
    if pn in style_ref.index:
        row = style_ref.loc[pn]
        return row["battingstyle"], row["bowlingstyle"], "full_name_exact"

    # Initial-based match
    for ref_name, row in style_ref.iterrows():
        if initial_match(pn, ref_name):
            return row["battingstyle"], row["bowlingstyle"], "full_name_initial"

    return "unknown", "unknown", "none"


In [14]:
base[["batting_style", "bowling_style", "style_source"]] = base["player"].apply(
    lambda x: pd.Series(infer_styles_from_names(x))
)


In [15]:
base["style_source"].value_counts()


style_source
full_name_initial    1827
none                 1071
full_name_exact       660
Name: count, dtype: int64

In [16]:
base[
    base["style_source"] != "none"
][["player","batting_style","bowling_style"]].head(15)

,player,batting_style,bowling_style
0,SP Narine,left-hand-bat,right-arm-offbreak
1,GJ Maxwell,right-hand-bat,right-arm-offbreak
2,Rashid Khan,right-hand-bat,legbreak-googly
3,N Pooran,left-hand-bat,None
4,AD Hales,right-hand-bat,right-arm-fast-medium
5,CA Lynn,right-hand-bat,slow-left-arm-orthodox
6,JJ Bumrah,right-hand-bat,right-arm-fast
7,JM Vince,right-hand-bat,right-arm-fast-medium
8,Mustafizur Rahman,left-hand-bat,left-arm-fast-medium
9,JC Buttler,right-hand-bat,None


In [17]:
base[
    base["batting_style"] == "unknown"
].sample(10)

,player,bat_rpa_raw,balls_faced_rpa,bowl_rpa_raw,bowl_balls,bat_wpa_raw,bowl_wpa_raw,balls_faced_wpa,balls_bowled,bat_rpa_eligible,...,bat_wpa_z,bowl_wpa_z,total_player_value,last_match_played,teams_represented,inferred_nationality,nationality_source,batting_style,bowling_style,style_source
899,DM Housego,-2.275597,16.0,0.000000,NaN,0.000000,0.0,NaN,NaN,False,...,0.000000,0.0,0.000000,2014-06-13,['Gloucestershire'],unknown,none,unknown,unknown,none
2047,AO Morgan,-19.810597,30.0,-2.245064,25.0,0.000000,0.0,NaN,NaN,False,...,0.000000,0.0,0.000000,2020-09-16,['Glamorgan'],unknown,none,unknown,unknown,none
2275,AA Chakraborty,-3.307677,3.0,7.348460,42.0,0.000000,0.0,NaN,NaN,False,...,0.000000,0.0,0.000000,2019-11-15,['Tripura'],Cyprus,full_name,unknown,unknown,none
2425,T Kohli,76.101940,328.0,-52.246080,168.0,0.276258,-0.0,417.0,0.0,True,...,-0.057674,0.0,-0.006187,2022-10-20,['Mizoram'],unknown,none,unknown,unknown,none
1616,Kengo Bam,-1.717582,3.0,-3.347590,24.0,0.000000,0.0,NaN,NaN,False,...,0.000000,0.0,0.000000,2021-11-09,['Arunachal Pradesh'],unknown,none,unknown,unknown,none
1365,JJ Athanaze,0.000000,NaN,-1.609351,19.0,0.000000,0.0,NaN,NaN,False,...,0.000000,0.0,0.000000,2014-08-09,['Antigua Hawksbills'],unknown,none,unknown,unknown,none
2138,Mohit Mittan,-10.697582,5.0,0.000000,NaN,0.000000,0.0,NaN,NaN,False,...,0.000000,0.0,0.000000,2022-10-22,['Puducherry'],India,full_name,unknown,unknown,none
1178,Hazrat Luqman,2.837530,1.0,-18.749315,39.0,0.000000,0.0,NaN,NaN,False,...,0.000000,0.0,0.000000,2023-02-09,['Dubai Capitals'],unknown,none,unknown,unknown,none
1271,Ashwani Kumar,0.000000,NaN,-24.942698,52.0,0.000000,0.0,NaN,NaN,False,...,0.000000,0.0,0.000000,2025-06-01,"['Mumbai Indians', 'Punjab']",India,full_name,unknown,unknown,none
2808,Punit Bisht,1.889330,112.0,-11.043252,6.0,0.238406,-0.0,202.0,0.0,True,...,-0.084680,0.0,-0.394397,2021-11-09,['Meghalaya'],unknown,none,unknown,unknown,none


In [18]:
base.to_csv(
    "outputs/player_total_value_master_enriched.csv",
    index=False
)


another round for nationality

In [20]:
LISTA_NZ ={
    "Auckland Aces",
    "Auckland",
    "Canterbury",
    "Central Districts",
    "Northern Districts",
    "Otago Volts",
    "Otago",
    "Wellington Firebirds",
    "Wellington",
    "Nelson",
    "Southland",
    "New Zealand Army",
    "New Zealand Services"
}

INTERNATIONAL_TEAMS = {
    "Afghanistan",
    "Australia",
    "Bangladesh",
    "England",
    "India",
    "Ireland",
    "New Zealand",
    "Pakistan",
    "South Africa",
    "Sri Lanka",
    "West Indies",
    "Zimbabwe",
}

SMAT_TEAMS = {
    "Andhra",
    "Arunachal Pradesh",
    "Assam",
    "Baroda",
    "Bengal",
    "Bihar",
    "Chandigarh",
    "Chhattisgarh",
    "Delhi",
    "Goa",
    "Gujarat",
    "Haryana",
    "Himachal",
    "Himachal Pradesh",
    "Hyderabad (India)",
    "Jammu & Kashmir",
    "Jharkhand",
    "Karnataka",
    "Kerala",
    "Madhya Pradesh",
    "Maharashtra",
    "Manipur",
    "Meghalaya",
    "Mizoram",
    "Mumbai",
    "Nagaland",
    "Odisha",
    "Puducherry",
    "Punjab",
    "Railways",
    "Rajasthan",
    "Saurashtra",
    "Services",
    "Sikkim",
    "Tamil Nadu",
    "Tripura",
    "Uttar Pradesh",
    "Uttarakhand",
    "Vidarbha"
}



In [30]:
import pandas as pd

df = pd.read_csv("outputs/player_total_value_master_enriched.csv")


In [39]:
DOMESTIC_ANCHORS = {}

for t in SMAT_TEAMS:
    DOMESTIC_ANCHORS[normalize_team_name(t)] = "India"

for t in LISTA_NZ:
    DOMESTIC_ANCHORS[normalize_team_name(t)] = "New Zealand"


In [40]:
def parse_and_normalize_teams(x):
    if pd.isna(x):
        return set()

    # handle list-like strings
    if x.startswith("[") and x.endswith("]"):
        x = x.strip("[]").replace("'", "").replace('"', "")

    if "|" in x:
        parts = x.split("|")
    elif "," in x:
        parts = x.split(",")
    else:
        parts = [x]

    return {normalize_team_name(p.strip()) for p in parts if p.strip()}


In [41]:
df["teams_list_norm"] = df["teams_represented"].apply(parse_and_normalize_teams)


In [42]:
df

,player,bat_rpa_raw,balls_faced_rpa,bowl_rpa_raw,bowl_balls,bat_wpa_raw,bowl_wpa_raw,balls_faced_wpa,balls_bowled,bat_rpa_eligible,...,total_player_value,last_match_played,teams_represented,inferred_nationality,nationality_source,batting_style,bowling_style,style_source,teams_list,teams_list_norm
0,SP Narine,330.255438,1520.0,1201.591867,5445.0,-0.934508,11.405882,1409.0,4499.0,True,...,25.382896,2025-07-05,"['Abu Dhabi Knight Riders', 'Cape Cobras', 'Co...",West Indies,spaced_initials,left-hand-bat,right-arm-offbreak,full_name_initial,"{West Indies, Melbourne Renegades, Comilla Vic...","{surrey, abu dhabi knight riders, comilla vict..."
1,GJ Maxwell,926.283009,3019.0,-61.816390,2010.0,7.087060,-1.010478,2974.0,1446.0,True,...,12.052219,2025-07-26,"['Australia', 'Birmingham Bears', 'Hampshire',...",Australia,spaced_initials,right-hand-bat,right-arm-offbreak,full_name_initial,"{Yorkshire, Australia, Royal Challengers Benga...","{lancashire, hampshire, punjab kings, washingt..."
2,Rashid Khan,105.502166,588.0,481.206109,3890.0,-1.907995,7.776876,559.0,3759.0,True,...,11.314485,2025-05-30,"['Adelaide Strikers', 'Barbados Royals', 'Comi...",Afghanistan,spaced_initials,right-hand-bat,legbreak-googly,full_name_exact,"{Trent Rockets, Comilla Victorians, Sunrisers ...","{adelaide strikers, comilla victorians, st kit..."
3,N Pooran,747.722923,3489.0,0.000000,NaN,6.116756,-0.000000,3024.0,0.0,True,...,10.991499,2025-07-13,"['Barbados Royals', ""Durban'S Super Giants"", '...",West Indies,spaced_initials,left-hand-bat,NaN,full_name_initial,"{Lucknow Super Giants, Northern Superchargers,...","{durbans super giants, rangpur riders, punjab ..."
4,AD Hales,254.911179,3252.0,0.000000,NaN,11.022656,-0.000000,4586.0,0.0,True,...,9.739729,2025-07-06,"['Adelaide Strikers', 'Barbados Royals', 'Dese...",England,spaced_initials,right-hand-bat,right-arm-fast-medium,full_name_initial,"{Desert Vipers, Nottinghamshire, Trent Rockets...","{rangpur riders, durban heat, islamabad united..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3553,Anwar Ali,-11.916317,101.0,-201.719062,816.0,-1.160317,-1.796420,232.0,452.0,True,...,-5.124264,2024-03-11,"['Dambulla Viiking', 'Galle Titans', 'Karachi ...",Pakistan,spaced_initials,right-hand-bat,right-arm-fast-medium,full_name_exact,"{Dambulla Viiking, Quetta Gladiators, Karachi ...","{rangpur riders, quetta gladiators, multan sul..."
3554,L Gregory,72.445111,1010.0,-330.190864,1964.0,0.512653,-2.742700,1025.0,1351.0,True,...,-5.370107,2025-07-18,"['Brisbane Heat', 'England', 'Islamabad United...",England,spaced_initials,right-hand-bat,right-arm-fast-medium,full_name_initial,"{Trent Rockets, Sharjah Warriors, Joburg Super...","{islamabad united, brisbane heat, karachi king..."
3555,HH Pandya,56.272101,2078.0,-257.532845,1587.0,-0.768185,-2.941751,1153.0,1842.0,True,...,-5.790200,2025-06-01,"['Baroda', 'Gujarat Titans', 'India', 'Mumbai ...",India,spaced_initials,right-hand-bat,right-arm-fast-medium,full_name_initial,"{Baroda, Mumbai Indians, India, Gujarat Titans}","{baroda, gujarat titans, mumbai indians, india}"
3556,KMA Paul,-53.248839,474.0,-301.917731,760.0,-0.274577,-1.821675,374.0,657.0,True,...,-6.041781,2025-01-19,"['Colombo Strikers', 'Delhi Capitals', ""Durban...",West Indies,spaced_initials,right-hand-bat,right-arm-fast-medium,full_name_initial,"{West Indies, Hobart Hurricanes, Guyana Amazon...","{durbans super giants, st lucia kings, hobart ..."


In [43]:
def infer_country_from_teams(teams):
    intl = {normalize_team_name(t) for t in INTERNATIONAL_TEAMS}
    intl_hit = teams & intl
    if len(intl_hit) == 1:
        return list(intl_hit)[0].title(), "international_team"

    domestic_hits = {DOMESTIC_ANCHORS[t] for t in teams if t in DOMESTIC_ANCHORS}
    if len(domestic_hits) == 1:
        return list(domestic_hits)[0], "domestic_anchor"

    return None, None


In [44]:
HIGH_CONF_SOURCES = {
    "spaced_initials",
    "full_name",
    "international_team"
}

def backfill_nationality(row):
    if pd.notna(row["nationality_source"]) and row["nationality_source"] in HIGH_CONF_SOURCES:
        return row["inferred_nationality"], row["nationality_source"]

    country, source = infer_country_from_teams(row["teams_list_norm"])
    if country is not None:
        return country, source

    return row["inferred_nationality"], row["nationality_source"]


In [45]:
df[["inferred_nationality", "nationality_source"]] = df.apply(
    lambda r: pd.Series(backfill_nationality(r)),
    axis=1
)


In [46]:
print(df["nationality_source"].value_counts())


nationality_source
spaced_initials       1471
full_name             1356
domestic_anchor        511
none                   204
international_team      16
Name: count, dtype: int64


In [47]:
df_debug = df[
    (df["nationality_source"] == "none") &
    (df["teams_list_norm"].apply(lambda x: len(x & set(DOMESTIC_ANCHORS.keys())) > 0))
]

df_debug.head(10)


,player,bat_rpa_raw,balls_faced_rpa,bowl_rpa_raw,bowl_balls,bat_wpa_raw,bowl_wpa_raw,balls_faced_wpa,balls_bowled,bat_rpa_eligible,...,total_player_value,last_match_played,teams_represented,inferred_nationality,nationality_source,batting_style,bowling_style,style_source,teams_list,teams_list_norm


In [48]:
df.to_csv(
    "outputs/player_total_value_master_enriched_v2.csv",
    index=False
)


In [49]:
# Columns safe to drop
DROP_COLS = [
    "teams_list",
    "teams_list_norm"
]

df_final = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

df_final.to_csv(
    "outputs/player_total_value_master_enriched_final.csv",
    index=False
)


In [50]:
print(df_final.columns.tolist())
print(df_final["nationality_source"].value_counts())


['player', 'bat_rpa_raw', 'balls_faced_rpa', 'bowl_rpa_raw', 'bowl_balls', 'bat_wpa_raw', 'bowl_wpa_raw', 'balls_faced_wpa', 'balls_bowled', 'bat_rpa_eligible', 'bowl_rpa_eligible', 'bat_wpa_eligible', 'bowl_wpa_eligible', 'bat_rpa_z', 'bowl_rpa_z', 'bat_wpa_z', 'bowl_wpa_z', 'total_player_value', 'last_match_played', 'teams_represented', 'inferred_nationality', 'nationality_source', 'batting_style', 'bowling_style', 'style_source']
nationality_source
spaced_initials       1471
full_name             1356
domestic_anchor        511
none                   204
international_team      16
Name: count, dtype: int64
